# DataInsight — Time series: Dự báo lưu lượng giao thông theo giờ trên cao tốc I-94

| | |
|---|---|
| **Học phần** | Nền tảng lập trình cho phân tích và trực quan dữ liệu (CO5177) — HK 261, 2026–2027 |
| **Giảng viên** | Lê Thành Sách — Khoa KH&KT Máy tính, ĐH Bách Khoa – ĐHQG-HCM |
| **Nhóm** | DataInsight |
| **Phụ trách bài Time series** | Trương Thanh Minh (2580893) |
| **Dataset** | [Metro Interstate Traffic Volume](https://www.kaggle.com/datasets/pooriamst/metro-interstate-traffic-volume) — Kaggle, bản gốc UCI, giấy phép CC BY 4.0 |

**Bài toán.** Dự báo `traffic_volume` (số xe/giờ đi hướng tây trên I-94, đoạn Minneapolis – St Paul) từ lịch sử lưu lượng, lịch (giờ, thứ, ngày lễ) và thời tiết.

Notebook này gồm **phần 1 – mô tả dữ liệu, EDA và chuẩn bị dữ liệu**. Phần mô hình hóa (baseline naive, Ridge, Random Forest, Gradient Boosting) sẽ nối tiếp ngay sau mục 5, dùng lại các biến `X_train`, `X_val`, `X_test` tạo ở cuối notebook.

**Mục lục**
1. [Cài đặt và tải dữ liệu](#1)
2. [Mô tả tập dữ liệu](#2)
3. [Kiểm tra chất lượng dữ liệu](#3)
4. [Làm sạch: gộp trùng, chọn giai đoạn, reindex và nội suy](#4)
5. [EDA trực quan](#5)
6. [Chuẩn bị dữ liệu cho mô hình: đặc trưng, chia theo thời gian, encoding, scaling](#6)
7. [Tổng kết EDA](#7)

> Notebook chạy được bằng **Runtime → Run all** trên Google Colab: dữ liệu tải thẳng từ UCI Machine Learning Repository, không cần Kaggle API key hay mount Drive.

<a id="1"></a>
## 1. Cài đặt và tải dữ liệu

Colab đã có sẵn `pandas`, `numpy`, `scikit-learn`; ô dưới chỉ bảo đảm `plotly` và `statsmodels` đủ mới (cần `MSTL` của statsmodels ≥ 0.14).

In [1]:
%pip install -q "plotly>=5.15" "statsmodels>=0.14"

Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 26.2.1 -> 26.2.2
[notice] To update, run: C:\Users\trthm\AppData\Local\Temp\claude\D--PhD-Nentanglaptrinh\731d40fd-9738-48ee-a3cb-66d24e2cf32d\scratchpad\venv\Scripts\python.exe -m pip install --upgrade pip


In [2]:
import warnings

import numpy as np
import pandas as pd
import plotly.express as px
import plotly.graph_objects as go
import plotly.io as pio
from plotly.subplots import make_subplots
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from statsmodels.tsa.seasonal import MSTL
from statsmodels.tsa.stattools import acf, pacf

warnings.filterwarnings("ignore")
SEED = 42
np.random.seed(SEED)
pd.set_option("display.max_columns", 50)
pd.set_option("display.float_format", "{:,.2f}".format)

# Bảng màu dùng chung cho mọi biểu đồ: màu theo vai trò, cố định thứ tự.
BLUE, ORANGE, AQUA, YELLOW = "#2a78d6", "#eb6834", "#1baf7a", "#eda100"
RED, MUTED, GRID = "#e34948", "#898781", "#e1e0d9"
SEQ_BLUE = ["#cde2fb", "#9ec5f4", "#6da7ec", "#3987e5", "#256abf", "#184f95", "#0d366b"]
DIVERGING = [[0, BLUE], [0.5, "#f0efec"], [1, RED]]

pio.templates["datainsight"] = go.layout.Template(layout=dict(
    font=dict(family="Inter, Segoe UI, Arial, sans-serif", size=13, color="#0b0b0b"),
    colorway=[BLUE, ORANGE, AQUA, YELLOW],
    paper_bgcolor="#ffffff", plot_bgcolor="#ffffff",
    title=dict(x=0.01, font=dict(size=16)),
    xaxis=dict(gridcolor=GRID, linecolor="#c9c8c3", zeroline=False, title_font_color="#52514e"),
    yaxis=dict(gridcolor=GRID, linecolor="#c9c8c3", zeroline=False, title_font_color="#52514e"),
    legend=dict(orientation="h", yanchor="bottom", y=1.02, x=0, title_text=""),
    margin=dict(l=60, r=20, t=70, b=50),
))
pio.templates.default = "plotly_white+datainsight"

DAY_VI = ["T2", "T3", "T4", "T5", "T6", "T7", "CN"]


def hist_bar(values, bins, color=BLUE, name=None):
    # Histogram tính sẵn bằng numpy: notebook nhẹ hơn nhiều so với nhúng toàn bộ điểm dữ liệu.
    counts, edges = np.histogram(values[~np.isnan(values)], bins=bins)
    centers = (edges[:-1] + edges[1:]) / 2
    return go.Bar(x=centers, y=counts, width=np.diff(edges) * 0.92, marker_color=color, name=name,
                  hovertemplate="%{x:,.1f}: %{y:,} giờ<extra></extra>")

In [3]:
DATA_URL = "https://archive.ics.uci.edu/ml/machine-learning-databases/00492/Metro_Interstate_Traffic_Volume.csv.gz"

# keep_default_na=False: giữ chuỗi "None" của cột holiday, không để pandas đổi thành NaN.
raw = pd.read_csv(DATA_URL, compression="gzip", keep_default_na=False, na_values=[])
print("Đã tải dữ liệu từ:", DATA_URL)
raw["date_time"] = pd.to_datetime(raw["date_time"])
print(f"Kích thước: {raw.shape[0]:,} dòng × {raw.shape[1]} cột")
raw.head()

Đã tải dữ liệu từ: https://archive.ics.uci.edu/ml/machine-learning-databases/00492/Metro_Interstate_Traffic_Volume.csv.gz
Kích thước: 48,204 dòng × 9 cột


,holiday,temp,rain_1h,snow_1h,clouds_all,weather_main,weather_description,date_time,traffic_volume
0,None,288.28,0.00,0.00,40,Clouds,scattered clouds,2012-10-02 09:00:00,5545
1,None,289.36,0.00,0.00,75,Clouds,broken clouds,2012-10-02 10:00:00,4516
2,None,289.58,0.00,0.00,90,Clouds,overcast clouds,2012-10-02 11:00:00,4767
3,None,290.13,0.00,0.00,90,Clouds,overcast clouds,2012-10-02 12:00:00,5026
4,None,291.14,0.00,0.00,75,Clouds,broken clouds,2012-10-02 13:00:00,4918


<a id="2"></a>
## 2. Mô tả tập dữ liệu

Dữ liệu ghi lưu lượng xe theo giờ tại trạm đếm ATR 301 của Sở Giao thông Minnesota (MnDOT), nằm khoảng giữa Minneapolis và St Paul, kèm thời tiết theo giờ (OpenWeatherMap) và ngày lễ. Khoảng thời gian: 10/2012 – 09/2018.

| Cột | Kiểu | Ý nghĩa |
|---|---|---|
| `date_time` | datetime | Mốc giờ (giờ địa phương CST) |
| `traffic_volume` | số nguyên | **Biến mục tiêu** — số xe đi hướng tây trong giờ đó |
| `holiday` | phân loại | Tên ngày lễ quốc gia/vùng (`None` nếu không phải ngày lễ) |
| `temp` | số thực | Nhiệt độ trung bình (Kelvin) |
| `rain_1h`, `snow_1h` | số thực | Lượng mưa / tuyết trong giờ (mm) |
| `clouds_all` | số nguyên | Độ che phủ mây (%) |
| `weather_main` | phân loại | Nhóm thời tiết (Clear, Clouds, Rain, Snow, ...) |
| `weather_description` | phân loại | Mô tả thời tiết chi tiết hơn |

In [4]:
raw.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 48204 entries, 0 to 48203
Data columns (total 9 columns):
 #   Column               Non-Null Count  Dtype         
---  ------               --------------  -----         
 0   holiday              48204 non-null  object        
 1   temp                 48204 non-null  float64       
 2   rain_1h              48204 non-null  float64       
 3   snow_1h              48204 non-null  float64       
 4   clouds_all           48204 non-null  int64         
 5   weather_main         48204 non-null  object        
 6   weather_description  48204 non-null  object        
 7   date_time            48204 non-null  datetime64[ns]
 8   traffic_volume       48204 non-null  int64         
dtypes: datetime64[ns](1), float64(3), int64(2), object(3)
memory usage: 3.3+ MB


In [5]:
raw.describe(include="all").T

,count,unique,top,freq,mean,min,25%,50%,75%,max,std
holiday,48204,12,None,48143,NaN,NaN,NaN,NaN,NaN,NaN,NaN
temp,"48,204.00",NaN,NaN,NaN,281.21,0.00,272.16,282.45,291.81,310.07,13.34
rain_1h,"48,204.00",NaN,NaN,NaN,0.33,0.00,0.00,0.00,0.00,"9,831.30",44.79
snow_1h,"48,204.00",NaN,NaN,NaN,0.00,0.00,0.00,0.00,0.00,0.51,0.01
clouds_all,"48,204.00",NaN,NaN,NaN,49.36,0.00,1.00,64.00,90.00,100.00,39.02
weather_main,48204,11,Clouds,15164,NaN,NaN,NaN,NaN,NaN,NaN,NaN
weather_description,48204,38,sky is clear,11665,NaN,NaN,NaN,NaN,NaN,NaN,NaN
date_time,48204,NaN,NaN,NaN,2016-01-05 10:46:16.773711616,2012-10-02 09:00:00,2014-02-06 11:45:00,2016-06-11 03:30:00,2017-08-11 06:00:00,2018-09-30 23:00:00,NaN
traffic_volume,"48,204.00",NaN,NaN,NaN,"3,259.82",0.00,"1,193.00","3,380.00","4,933.00","7,280.00","1,986.86"


In [6]:
for col in ["holiday", "weather_main"]:
    print(f"--- {col}: {raw[col].nunique()} giá trị")
    print(raw[col].value_counts().to_string(), "\n")
print("weather_description:", raw["weather_description"].nunique(), "giá trị")

--- holiday: 12 giá trị
holiday
None                         48143
Labor Day                        7
Christmas Day                    6
Thanksgiving Day                 6
Martin Luther King Jr Day        6
New Years Day                    6
Veterans Day                     5
Columbus Day                     5
Memorial Day                     5
Washingtons Birthday             5
State Fair                       5
Independence Day                 5 

--- weather_main: 11 giá trị
weather_main
Clouds          15164
Clear           13391
Mist             5950
Rain             5672
Snow             2876
Drizzle          1821
Haze             1360
Thunderstorm     1034
Fog               912
Smoke              20
Squall              4 

weather_description: 38 giá trị


<a id="3"></a>
## 3. Kiểm tra chất lượng dữ liệu

Dữ liệu chuỗi thời gian theo giờ cần đúng **một dòng cho mỗi giờ, không thiếu giờ nào**. Mục này kiểm tra bốn vấn đề: mốc giờ trùng, mốc giờ bị thiếu, giá trị cảm biến vô lý và cách gán nhãn ngày lễ.

### 3.1 Mốc giờ bị trùng

In [7]:
dup_mask = raw.duplicated("date_time", keep=False)
n_dup = raw.duplicated("date_time").sum()
print(f"Số dòng thừa do trùng mốc giờ: {n_dup:,} / {len(raw):,} ({n_dup / len(raw):.1%})")
print(f"Số mốc giờ duy nhất: {raw['date_time'].nunique():,}")

# Trong mỗi nhóm trùng, cột nào thực sự khác nhau?
n_distinct = raw[dup_mask].groupby("date_time").nunique()
print("\nTỷ lệ nhóm trùng có giá trị khác nhau theo từng cột:")
print((n_distinct > 1).mean().map("{:.1%}".format).to_string())

example = raw.loc[raw["date_time"] == raw.loc[dup_mask, "date_time"].value_counts().idxmax()]
example

Số dòng thừa do trùng mốc giờ: 7,629 / 48,204 (15.8%)
Số mốc giờ duy nhất: 40,575

Tỷ lệ nhóm trùng có giá trị khác nhau theo từng cột:
holiday                 0.0%
temp                    1.4%
rain_1h                 0.2%
snow_1h                 0.0%
clouds_all              0.6%
weather_main           98.2%
weather_description    98.9%
traffic_volume          0.0%


,holiday,temp,rain_1h,snow_1h,clouds_all,weather_main,weather_description,date_time,traffic_volume
6162,None,287.15,0.00,0.00,90,Thunderstorm,thunderstorm with light rain,2013-05-19 10:00:00,3591
6163,None,287.15,0.00,0.00,90,Mist,mist,2013-05-19 10:00:00,3591
6164,None,287.15,0.00,0.00,90,Thunderstorm,proximity thunderstorm,2013-05-19 10:00:00,3591
6165,None,287.15,0.00,0.00,90,Rain,moderate rain,2013-05-19 10:00:00,3591
6166,None,287.15,0.00,0.00,90,Rain,light rain,2013-05-19 10:00:00,3591
6167,None,287.15,0.00,0.00,90,Thunderstorm,thunderstorm with heavy rain,2013-05-19 10:00:00,3591


**Nhận xét.** Các dòng trùng chỉ khác nhau ở `weather_main` / `weather_description`: khi trong một giờ có nhiều hiện tượng thời tiết (vd. vừa mưa nhỏ vừa sương mù), nguồn dữ liệu ghi mỗi hiện tượng thành một dòng và **lặp lại nguyên `traffic_volume`**. Nếu giữ nguyên, các giờ thời tiết phức tạp bị đếm nhiều lần và chuỗi không còn đều theo giờ. → Ở mục 4 ta **gộp về một dòng mỗi giờ**, giữ hiện tượng đầu tiên và thêm cột `n_weather` (số hiện tượng thời tiết trong giờ) để không mất thông tin.

### 3.2 Mốc giờ bị thiếu

In [8]:
hourly_raw = raw.drop_duplicates("date_time").set_index("date_time").sort_index().asfreq("h")
is_missing = hourly_raw["traffic_volume"].isna()
print(f"Từ {hourly_raw.index.min()} đến {hourly_raw.index.max()}")
print(f"Số giờ theo lịch: {len(hourly_raw):,} | có dữ liệu: {(~is_missing).sum():,} | "
      f"thiếu: {is_missing.sum():,} ({is_missing.mean():.1%})")


def gap_table(missing):
    # Mỗi đoạn liên tiếp các giờ thiếu là một "gap".
    run_id = missing.ne(missing.shift()).cumsum()[missing]
    idx = pd.Series(missing.index[missing], index=run_id.values)
    return (idx.groupby(level=0).agg(start="min", end="max", hours="size")
              .sort_values("hours", ascending=False).reset_index(drop=True))


gaps_all = gap_table(is_missing)
print(f"Số đoạn bị thiếu: {len(gaps_all):,}")
gaps_all.head(6)

Từ 2012-10-02 09:00:00 đến 2018-09-30 23:00:00
Số giờ theo lịch: 52,551 | có dữ liệu: 40,575 | thiếu: 11,976 (22.8%)


Số đoạn bị thiếu: 2,588


,start,end,hours
0,2014-08-08 02:00:00,2015-06-11 19:00:00,7386
1,2013-10-27 02:00:00,2013-11-06 03:00:00,242
2,2015-06-14 21:00:00,2015-06-19 17:00:00,117
3,2014-04-29 09:00:00,2014-05-04 04:00:00,116
4,2015-10-23 12:00:00,2015-10-27 07:00:00,92
5,2014-07-27 17:00:00,2014-07-30 08:00:00,64


In [9]:
daily_raw = hourly_raw["traffic_volume"].resample("D").mean()
big = gaps_all.iloc[0]

fig = go.Figure(go.Scatter(x=daily_raw.index, y=daily_raw.values, mode="lines", line=dict(color=BLUE, width=1.5),
                           name="Trung bình ngày", hovertemplate="%{x|%d/%m/%Y}: %{y:,.0f} xe/giờ<extra></extra>"))
fig.add_vrect(x0=big.start, x1=big.end, fillcolor=ORANGE, opacity=0.12, line_width=0,
              annotation_text=f"Thiếu liên tục {big.hours:,} giờ (~{big.hours / 24:.0f} ngày)",
              annotation_position="top left", annotation_font_color="#52514e")
fig.add_vline(x=pd.Timestamp("2016-01-01"), line_dash="dot", line_color=MUTED)
fig.add_annotation(x=pd.Timestamp("2016-01-01"), y=1.0, yref="paper", text="Giai đoạn chọn để mô hình hóa →",
                   showarrow=False, xanchor="left", font_color="#52514e")
fig.update_layout(title="Hình 1. Lưu lượng trung bình theo ngày, 2012–2018: dữ liệu bị đứt nhiều đoạn trước 2016",
                  yaxis_title="Xe/giờ (trung bình ngày)", height=420, showlegend=False)
fig.show()

In [10]:
coverage = (~is_missing).resample("MS").mean() * 100
fig = go.Figure(go.Bar(x=coverage.index, y=coverage.values,
                       marker_color=np.where(coverage.index >= "2016-01-01", BLUE, MUTED),
                       hovertemplate="%{x|%m/%Y}: %{y:.1f}% số giờ có dữ liệu<extra></extra>"))
fig.update_layout(title="Hình 2. Tỷ lệ giờ có dữ liệu theo tháng (xanh = giai đoạn 2016–2018 được chọn)",
                  yaxis_title="% số giờ có dữ liệu", yaxis_range=[0, 105], height=380, bargap=0.15)
fig.show()

coverage_by_period = pd.DataFrame({
    "Giai đoạn": ["10/2012 – 12/2015", "01/2016 – 09/2018"],
    "% giờ có dữ liệu": [100 * (~is_missing[:"2015-12-31"]).mean(), 100 * (~is_missing["2016-01-01":]).mean()],
    "Gap dài nhất (giờ)": [gap_table(is_missing[:"2015-12-31"])["hours"].max(),
                           gap_table(is_missing["2016-01-01":])["hours"].max()],
})
coverage_by_period

,Giai đoạn,% giờ có dữ liệu,Gap dài nhất (giờ)
0,10/2012 – 12/2015,61.47,7386
1,01/2016 – 09/2018,95.80,9


In [11]:
bins = [0, 1, 3, 6, 24, 168, np.inf]
labels = ["1 giờ", "2–3 giờ", "4–6 giờ", "7–24 giờ", "1–7 ngày", "> 7 ngày"]
gap_buckets = pd.cut(gaps_all["hours"], bins=bins, labels=labels).value_counts().reindex(labels)
fig = go.Figure(go.Bar(x=labels, y=gap_buckets.values, marker_color=BLUE, text=gap_buckets.values,
                       textposition="outside", textfont_color="#52514e",
                       hovertemplate="%{x}: %{y:,} đoạn<extra></extra>"))
fig.update_layout(title="Hình 3. Phân bố độ dài các đoạn bị thiếu (toàn bộ 2012–2018)",
                  xaxis_title="Độ dài đoạn thiếu", yaxis_title="Số đoạn", height=380)
fig.show()

**Nhận xét.** Phần lớn các đoạn thiếu chỉ dài 1 giờ (trạm đếm mất tín hiệu ngắn) — loại này nội suy an toàn. Nhưng trước 2016 có những đoạn thiếu kéo dài nhiều ngày đến gần **10 tháng** (08/2014 – 06/2015): không phương pháp nội suy nào phục hồi được, và lag theo tuần (`lag_168`) sẽ bị sai trên các đoạn đó.

→ **Quyết định:** dùng giai đoạn **01/01/2016 – 30/09/2018** (≈ 2 năm 9 tháng, đủ 2 chu kỳ năm cho mùa vụ). Trong giai đoạn này đoạn thiếu dài nhất chỉ vài giờ (bảng trên), nên reindex + nội suy theo thời gian là hợp lý.

### 3.3 Giá trị cảm biến vô lý

In [12]:
print("Nhiệt độ thấp nhất (K):", raw["temp"].min(), "→ 0 K = −273 °C, chắc chắn là lỗi cảm biến")
print(raw.loc[raw["temp"] <= 0, ["date_time", "temp", "weather_main"]].to_string(index=False), "\n")
print("Lượng mưa lớn nhất trong 1 giờ (mm):", raw["rain_1h"].max(), "→ kỷ lục thế giới chỉ khoảng 300 mm/giờ")
print(raw.nlargest(3, "rain_1h")[["date_time", "rain_1h", "weather_description"]].to_string(index=False))

Nhiệt độ thấp nhất (K): 0.0 → 0 K = −273 °C, chắc chắn là lỗi cảm biến
          date_time  temp weather_main
2014-01-31 03:00:00  0.00        Clear
2014-01-31 04:00:00  0.00        Clear
2014-01-31 05:00:00  0.00        Clear
2014-01-31 06:00:00  0.00        Clear
2014-02-02 03:00:00  0.00        Clear
2014-02-02 04:00:00  0.00        Clear
2014-02-02 05:00:00  0.00        Clear
2014-02-02 06:00:00  0.00        Clear
2014-02-02 07:00:00  0.00        Clear
2014-02-02 08:00:00  0.00        Clear 

Lượng mưa lớn nhất trong 1 giờ (mm): 9831.3 → kỷ lục thế giới chỉ khoảng 300 mm/giờ
          date_time  rain_1h weather_description
2016-07-11 17:00:00 9,831.30     very heavy rain
2013-08-07 02:00:00    55.63     very heavy rain
2013-06-24 11:00:00    44.45     very heavy rain


In [13]:
temp_c_raw = raw["temp"].to_numpy() - 273.15
fig = make_subplots(rows=1, cols=2, subplot_titles=("Nhiệt độ (°C) — có 10 giờ ở −273 °C", "Lượng mưa 1 giờ (mm, thang log)"))
fig.add_trace(hist_bar(temp_c_raw, bins=80, color=BLUE), row=1, col=1)
rain_pos = raw.loc[raw["rain_1h"] > 0, "rain_1h"].to_numpy()
fig.add_trace(hist_bar(np.log10(rain_pos), bins=60, color=AQUA), row=1, col=2)
fig.update_xaxes(title_text="°C", row=1, col=1)
fig.update_xaxes(title_text="log10(mm) — điểm lẻ ở bên phải ≈ 9.831 mm", row=1, col=2)
fig.update_yaxes(title_text="Số giờ (thang log)", type="log", dtick=1, row=1, col=1)
fig.update_yaxes(title_text="Số giờ (thang log)", type="log", dtick=1, row=1, col=2)
fig.update_layout(title="Hình 4. Hai cột thời tiết có giá trị ngoại lai do lỗi đo", showlegend=False, height=400)
fig.show()

**Nhận xét.** 10 giờ có `temp = 0 K` và 1 giờ có `rain_1h ≈ 9.831 mm` là **lỗi đo**, không phải hiện tượng thật → đổi thành `NaN` rồi nội suy như giờ bị thiếu. Ngoài ra đổi nhiệt độ sang °C cho dễ đọc. Các ngoại lai của chính `traffic_volume` (ví dụ đường bị đóng) là sự kiện có thật nên được xử lý riêng ở mục 5.7.

### 3.4 Nhãn ngày lễ chỉ gắn ở 00:00

In [14]:
holiday_rows = raw[raw["holiday"] != "None"]
print("Giờ trong ngày của các dòng có nhãn holiday:")
print(holiday_rows["date_time"].dt.hour.value_counts().to_string())
holiday_rows.head(8)[["date_time", "holiday", "traffic_volume"]]

Giờ trong ngày của các dòng có nhãn holiday:
date_time
0    61


,date_time,holiday,traffic_volume
126,2012-10-08,Columbus Day,455
1123,2012-11-12,Veterans Day,1000
1370,2012-11-22,Thanksgiving Day,919
2360,2012-12-25,Christmas Day,803
2559,2013-01-01,New Years Day,1439
3697,2013-02-18,Washingtons Birthday,556
6430,2013-05-27,Memorial Day,863
7414,2013-07-04,Independence Day,1060


**Nhận xét.** Nhãn ngày lễ chỉ xuất hiện ở dòng **00:00**, còn 23 giờ còn lại của cùng ngày ghi `None`. Nếu dùng nguyên, mô hình sẽ hiểu sai rằng "ngày lễ" chỉ kéo dài 1 giờ → ta **lan nhãn ra cả ngày** dựa vào ngày của mốc 00:00.

<a id="4"></a>
## 4. Làm sạch: gộp trùng, chọn giai đoạn, reindex và nội suy

Các bước, theo thứ tự:
1. Gộp các dòng trùng mốc giờ, thêm `n_weather`.
2. Đổi lỗi cảm biến (`temp ≤ 0 K`, `rain_1h > 100 mm`) thành `NaN`; đổi nhiệt độ sang °C.
3. Cắt giai đoạn 01/2016 – 09/2018 và **reindex theo tần suất 1 giờ** để lộ ra các giờ thiếu.
4. Đánh dấu `is_imputed = 1` cho giờ được điền, rồi **nội suy tuyến tính theo thời gian** các cột số. Lý do: đoạn thiếu dài nhất chỉ vài giờ, lưu lượng và thời tiết thay đổi liên tục nên nội suy tuyến tính sát thực tế hơn forward-fill (forward-fill sẽ "đóng băng" lưu lượng lúc giờ cao điểm bắt đầu tăng/giảm).
5. Cột phân loại (`weather_main`, `weather_description`) dùng **forward-fill**: thời tiết của giờ trước là ước lượng hợp lý nhất, và không thể nội suy giữa hai nhãn.
6. Lan nhãn ngày lễ ra cả ngày.

In [15]:
WINDOW_START, WINDOW_END = "2016-01-01 00:00", "2018-09-30 23:00"
NUM_COLS = ["traffic_volume", "temp_c", "rain_1h", "snow_1h", "clouds_all"]


def clean(raw):
    df = raw.copy()
    df["n_weather"] = df.groupby("date_time")["weather_main"].transform("size")
    df = df.drop_duplicates("date_time", keep="first").set_index("date_time").sort_index()

    df.loc[df["temp"] <= 0, "temp"] = np.nan
    df.loc[df["rain_1h"] > 100, "rain_1h"] = np.nan
    df["temp_c"] = df["temp"] - 273.15

    # Ngày lễ: lấy tên ngày lễ của mốc 00:00 rồi gán cho mọi giờ cùng ngày.
    holiday_by_day = df.loc[df["holiday"] != "None", "holiday"]
    holiday_by_day.index = holiday_by_day.index.normalize()

    df = df.loc[WINDOW_START:WINDOW_END].asfreq("h")
    df["is_imputed"] = df["traffic_volume"].isna().astype(int)
    df[NUM_COLS] = df[NUM_COLS].interpolate(method="time", limit_direction="both")
    df[["weather_main", "weather_description"]] = df[["weather_main", "weather_description"]].ffill().bfill()
    df["n_weather"] = df["n_weather"].fillna(1).astype(int)
    df["holiday"] = df.index.normalize().map(holiday_by_day).fillna("None")
    df["is_holiday"] = (df["holiday"] != "None").astype(int)
    return df.drop(columns="temp")[["traffic_volume", "is_imputed", "holiday", "is_holiday", "temp_c",
                                    "rain_1h", "snow_1h", "clouds_all", "weather_main",
                                    "weather_description", "n_weather"]]


ts = clean(raw)
y = ts["traffic_volume"]

summary = pd.DataFrame({
    "Bước": ["Dữ liệu gốc", "Sau khi gộp trùng (toàn bộ 2012–2018)", "Giờ có dữ liệu trong 2016–2018",
             "Sau reindex 2016–2018 (đủ mọi giờ)", "  trong đó: giờ được nội suy"],
    "Số dòng": [len(raw), raw["date_time"].nunique(), int((ts["is_imputed"] == 0).sum()),
                len(ts), int(ts["is_imputed"].sum())],
})
summary["% so với sau reindex"] = (summary["Số dòng"] / len(ts) * 100).round(1).astype(str) + "%"
summary.loc[[0, 1], "% so với sau reindex"] = "—"
print("Còn NaN sau làm sạch:", int(ts.isna().sum().sum()))
print("Số ngày lễ trong giai đoạn:", ts.loc[ts["is_holiday"] == 1].index.normalize().nunique())
summary

Còn NaN sau làm sạch: 0
Số ngày lễ trong giai đoạn: 28


,Bước,Số dòng,% so với sau reindex
0,Dữ liệu gốc,48204,—
1,Sau khi gộp trùng (toàn bộ 2012–2018),40575,—
2,Giờ có dữ liệu trong 2016–2018,23084,95.8%
3,Sau reindex 2016–2018 (đủ mọi giờ),24096,100.0%
4,trong đó: giờ được nội suy,1012,4.2%


In [16]:
gaps_win = gap_table(is_missing[WINDOW_START:WINDOW_END])
g = gaps_win.iloc[0]
lo, hi = g.start - pd.Timedelta(hours=36), g.end + pd.Timedelta(hours=36)
seg = ts.loc[lo:hi]
imp = seg[seg["is_imputed"] == 1]

fig = go.Figure()
fig.add_trace(go.Scatter(x=seg.index, y=seg["traffic_volume"], mode="lines", line=dict(color=BLUE, width=2),
                         name="Chuỗi sau làm sạch", hovertemplate="%{x|%d/%m %H:%M}: %{y:,.0f} xe<extra></extra>"))
fig.add_trace(go.Scatter(x=imp.index, y=imp["traffic_volume"], mode="markers",
                         marker=dict(color=ORANGE, size=9, line=dict(color="#ffffff", width=2)),
                         name="Giờ được nội suy", hovertemplate="%{x|%d/%m %H:%M}: %{y:,.0f} xe (nội suy)<extra></extra>"))
fig.update_layout(title=f"Hình 5. Đoạn thiếu dài nhất trong 2016–2018 ({g.hours} giờ) sau khi nội suy theo thời gian",
                  yaxis_title="Xe/giờ", height=400)
fig.show()

<a id="5"></a>
## 5. EDA trực quan

Từ đây mọi phân tích dùng chuỗi đã làm sạch `ts` (01/2016 – 09/2018, mỗi giờ một dòng).

### 5.1 Toàn cảnh chuỗi và một lát cắt 2 tuần

In [17]:
daily = y.resample("D").mean()
fig = go.Figure()
fig.add_trace(go.Scatter(x=daily.index, y=daily.values, mode="lines", line=dict(color=BLUE, width=1.2),
                         name="Trung bình ngày", hovertemplate="%{x|%d/%m/%Y}: %{y:,.0f}<extra></extra>"))
fig.add_trace(go.Scatter(x=daily.index, y=daily.rolling(28, center=True).mean().values, mode="lines",
                         line=dict(color=ORANGE, width=2.5), name="Trung bình trượt 28 ngày",
                         hovertemplate="%{x|%d/%m/%Y}: %{y:,.0f}<extra></extra>"))
fig.update_layout(title="Hình 6. Lưu lượng trung bình theo ngày 2016–2018: không có xu hướng tăng/giảm dài hạn rõ rệt",
                  yaxis_title="Xe/giờ (trung bình ngày)", height=430, hovermode="x unified",
                  xaxis=dict(rangeslider=dict(visible=True, thickness=0.06)))
fig.show()

In [18]:
week2 = ts.loc["2017-10-02":"2017-10-15"]
fig = go.Figure(go.Scatter(x=week2.index, y=week2["traffic_volume"], mode="lines", line=dict(color=BLUE, width=2),
                           hovertemplate="%{x|%a %d/%m %H:%M}: %{y:,.0f} xe<extra></extra>"))
for d in pd.date_range("2017-10-07", "2017-10-15", freq="7D"):
    fig.add_vrect(x0=d, x1=d + pd.Timedelta(days=2), fillcolor=MUTED, opacity=0.1, line_width=0)
fig.update_layout(title="Hình 7. Hai tuần 02–15/10/2017: hai đỉnh sáng – chiều ngày thường, cuối tuần (vùng xám) chỉ một đỉnh thấp",
                  yaxis_title="Xe/giờ", height=400, showlegend=False)
fig.show()

### 5.2 Phân phối của biến mục tiêu

In [19]:
vals = y.to_numpy()
fig = go.Figure(hist_bar(vals, bins=np.arange(0, 7600, 200), color=BLUE))
fig.add_vline(x=np.median(vals), line_dash="dot", line_color="#52514e",
              annotation_text=f"Trung vị {np.median(vals):,.0f}", annotation_font_color="#52514e")
fig.update_layout(title="Hình 8. Phân phối traffic_volume có hai đỉnh: giờ đêm (< 1.000) và giờ ban ngày (4.000–6.000)",
                  xaxis_title="Xe/giờ", yaxis_title="Số giờ", height=400, showlegend=False)
fig.show()
print(f"Tỷ lệ giờ < 1.000 xe: {(vals < 1000).mean():.1%} | 1.000–4.000: {((vals >= 1000) & (vals < 4000)).mean():.1%} "
      f"| ≥ 4.000: {(vals >= 4000).mean():.1%}")
print(f"Độ lệch chuẩn: {vals.std():,.0f} xe/giờ — lớn ~{vals.std() / vals.mean():.0%} so với trung bình")

Tỷ lệ giờ < 1.000 xe: 21.6% | 1.000–4.000: 33.5% | ≥ 4.000: 44.9%
Độ lệch chuẩn: 1,963 xe/giờ — lớn ~59% so với trung bình


**Nhận xét.** Phân phối **không chuẩn, có hai đỉnh**: khoảng 22% số giờ dưới 1.000 xe (đêm khuya) và 45% số giờ từ 4.000 xe trở lên (ban ngày). Độ lệch chuẩn bằng gần 60% trung bình — phần lớn biến động này đến từ giờ trong ngày chứ không phải nhiễu, nên mô hình **bắt buộc** phải có thông tin giờ (đặc trưng lịch hoặc lag 24).

### 5.3 Mùa vụ theo giờ trong ngày và thứ trong tuần

In [20]:
cal = pd.DataFrame({"y": y, "hour": y.index.hour, "dow": y.index.dayofweek, "month": y.index.month})
pivot = cal.pivot_table(index="dow", columns="hour", values="y", aggfunc="mean")
fig = go.Figure(go.Heatmap(z=pivot.values, x=[f"{h:02d}h" for h in pivot.columns], y=DAY_VI,
                           colorscale=SEQ_BLUE, colorbar=dict(title="Xe/giờ"), xgap=2, ygap=2,
                           hovertemplate="%{y} %{x}: %{z:,.0f} xe/giờ<extra></extra>"))
fig.update_layout(title="Hình 9. Lưu lượng trung bình theo giờ × thứ: cao điểm 7h và 16h chỉ có từ T2 đến T6",
                  yaxis_autorange="reversed", height=420)
fig.show()

In [21]:
monthly = cal.groupby("month")["y"].mean()
day_totals = y.resample("D").sum()
mth = pd.DataFrame({"total": day_totals, "month": day_totals.index.month})
fig = go.Figure()
for m in range(1, 13):
    fig.add_trace(go.Box(y=mth.loc[mth["month"] == m, "total"], name=f"Th{m}", marker_color=BLUE,
                         line_width=1.5, boxpoints="outliers", showlegend=False))
fig.update_layout(title="Hình 10. Tổng lưu lượng theo ngày, gom theo tháng: mùa vụ trong năm yếu hơn nhiều so với mùa vụ ngày/tuần",
                  yaxis_title="Tổng xe/ngày", height=420)
fig.show()
print("Chênh lệch trung bình giờ giữa tháng cao nhất và thấp nhất:",
      f"{(monthly.max() / monthly.min() - 1):.1%} (Th{monthly.idxmax()} vs Th{monthly.idxmin()})")
hour_mean = cal.groupby("hour")["y"].mean()
print("Chênh lệch giữa giờ cao nhất và thấp nhất:", f"{hour_mean.max() / hour_mean.min():.1f} lần "
      f"({hour_mean.idxmax()}h vs {hour_mean.idxmin()}h)")

Chênh lệch trung bình giờ giữa tháng cao nhất và thấp nhất: 11.8% (Th3 vs Th12)
Chênh lệch giữa giờ cao nhất và thấp nhất: 14.5 lần (16h vs 3h)


**Nhận xét.** Giờ cao nhất (16h) đông gấp **khoảng 14,5 lần** giờ thấp nhất (3h), trong khi tháng cao nhất chỉ hơn tháng thấp nhất khoảng **12%**. Heatmap cho thấy hai cao điểm 6–7h và 15–17h chỉ xuất hiện từ T2 đến T6 — đặc trưng của lưu lượng đi làm. → Chu kỳ **ngày** và **tuần** là quan trọng nhất; chu kỳ năm chỉ là hiệu chỉnh nhỏ (đặc trưng `month`).

### 5.4 Câu hỏi: ngày lễ làm lưu lượng giảm bao nhiêu?

In [22]:
day_type = np.select([ts["is_holiday"] == 1, ts.index.dayofweek >= 5], ["Ngày lễ", "Cuối tuần"], "Ngày thường (T2–T6)")
prof = (pd.DataFrame({"y": y, "hour": y.index.hour, "type": day_type})
          .groupby(["type", "hour"])["y"].mean().unstack(0))
order = ["Ngày thường (T2–T6)", "Cuối tuần", "Ngày lễ"]
fig = go.Figure()
for name, color in zip(order, [BLUE, ORANGE, AQUA]):
    fig.add_trace(go.Scatter(x=prof.index, y=prof[name], mode="lines+markers", name=name,
                             line=dict(color=color, width=2), marker=dict(size=8, line=dict(color="#ffffff", width=2)),
                             hovertemplate=f"{name} %{{x}}h: %{{y:,.0f}} xe<extra></extra>"))

daily_by_type = (pd.DataFrame({"y": y, "date": y.index.normalize(), "type": day_type})
                   .groupby(["type", "date"])["y"].sum().groupby("type").mean().reindex(order))
holiday_drop = 1 - daily_by_type["Ngày lễ"] / daily_by_type["Ngày thường (T2–T6)"]
weekend_drop = 1 - daily_by_type["Cuối tuần"] / daily_by_type["Ngày thường (T2–T6)"]
fig.update_layout(title=f"Hình 11. Hồ sơ lưu lượng theo giờ: ngày lễ thấp hơn ngày thường {holiday_drop:.0%}, "
                        f"cuối tuần thấp hơn {weekend_drop:.0%} (tổng xe/ngày)",
                  xaxis_title="Giờ trong ngày", yaxis_title="Xe/giờ (trung bình)", height=430, hovermode="x unified")
fig.show()
daily_by_type.rename("Tổng xe/ngày (trung bình)").to_frame().assign(
    **{"So với ngày thường": lambda d: (d.iloc[:, 0] / d.iloc[0, 0] - 1).map("{:+.1%}".format)})

,Tổng xe/ngày (trung bình),So với ngày thường
type,,
Ngày thường (T2–T6),"86,656.97",+0.0%
Cuối tuần,"64,092.21",-26.0%
Ngày lễ,"62,317.48",-28.1%


**Kết luận.** Tổng lưu lượng một **ngày lễ thấp hơn ngày thường khoảng 28%**, gần bằng mức giảm của cuối tuần (26%), và cao điểm sáng của ngày lễ chỉ còn khoảng một nửa ngày thường (≈ 3.100 so với ≈ 6.150 xe lúc 7h); buổi chiều thì ngày lễ gần giống cuối tuần. → Biến `is_holiday` là cần thiết: nếu thiếu, mô hình sẽ dự báo một ngày lễ rơi vào thứ Hai như một thứ Hai bình thường và sai khoảng 1/4 lưu lượng.

### 5.5 Câu hỏi: thời tiết xấu làm lưu lượng giảm bao nhiêu?

So sánh thẳng trung bình theo `weather_main` sẽ bị nhiễu: ví dụ sương mù (`Fog`, `Mist`) hay xảy ra lúc rạng sáng — giờ vốn vắng xe. Để công bằng, mỗi giờ được so với **trung vị của chính khung giờ đó** (cùng thứ, cùng giờ, ngày không lễ):

$$\text{chênh lệch} = \frac{y_t - \text{median}(y \mid \text{thứ}, \text{giờ})}{\text{median}(y \mid \text{thứ}, \text{giờ})}$$

Chỉ dùng giờ ban ngày 6h–20h (mẫu số đủ lớn) và giờ có dữ liệu thật (không nội suy).

In [23]:
base = ts[(ts["is_holiday"] == 0) & (ts["is_imputed"] == 0)].copy()
base["how"] = base.index.dayofweek * 24 + base.index.hour
base["expected"] = base.groupby("how")["traffic_volume"].transform("median")
day = base[(base.index.hour >= 6) & (base.index.hour <= 20)]
day = day.assign(rel=(day["traffic_volume"] / day["expected"] - 1) * 100)

eff = (day.groupby("weather_main")["rel"].agg(["mean", "count"]).rename(columns={"count": "n"})
          .loc[lambda d: d["n"] >= 30].sort_values("mean"))
fig = go.Figure(go.Bar(x=eff["mean"], y=eff.index, orientation="h",
                       marker_color=np.where(eff["mean"] < 0, RED, BLUE),
                       text=[f"{m:+.1f}% · n={n:,}" for m, n in zip(eff["mean"], eff["n"])],
                       textposition="outside", textfont_color="#52514e",
                       hovertemplate="%{y}: %{x:+.1f}% so với cùng khung giờ<extra></extra>"))
fig.add_vline(x=0, line_color="#52514e", line_width=1)
fig.update_layout(title="Hình 12. Chênh lệch lưu lượng so với cùng thứ-cùng giờ, theo loại thời tiết (6h–20h)",
                  xaxis_title="% chênh lệch trung bình", height=430, showlegend=False,
                  xaxis_range=[eff["mean"].min() * 1.6 - 1, max(eff["mean"].max() * 1.6, 2)])
fig.show()
eff.rename(columns={"mean": "% chênh lệch", "n": "Số giờ"})

,% chênh lệch,Số giờ
weather_main,,
Snow,-10.63,879
Thunderstorm,-3.04,210
Mist,-1.87,907
Rain,-1.81,1645
Haze,-0.67,290
Clear,-0.42,4172
Drizzle,-0.26,196
Clouds,-0.17,5577
Fog,-0.02,66


In [24]:
snow_rel = day.loc[day["weather_main"] == "Snow", "rel"]
heavy = day["weather_description"].str.contains("heavy", case=False)
print(f"Giờ có tuyết: {snow_rel.mean():+.1f}% (n={len(snow_rel):,})")
print(f"Giờ có hiện tượng 'heavy' (mưa/tuyết lớn): {day.loc[heavy, 'rel'].mean():+.1f}% (n={heavy.sum():,})")
print(f"Giờ trời quang (Clear): {day.loc[day['weather_main'] == 'Clear', 'rel'].mean():+.1f}%")

Giờ có tuyết: -10.6% (n=879)
Giờ có hiện tượng 'heavy' (mưa/tuyết lớn): -13.6% (n=201)
Giờ trời quang (Clear): -0.4%


**Kết luận.** Sau khi khử ảnh hưởng của giờ và thứ, phần lớn loại thời tiết chỉ làm lưu lượng lệch dưới 2% — gần như không đáng kể. Ngoại lệ rõ rệt là **tuyết: −10,6%** (879 giờ) và các giờ mưa/tuyết **"heavy": −13,6%**. → Thời tiết chỉ quan trọng ở các tình huống cực đoan; kỳ vọng các đặc trưng thời tiết đóng góp ít cho mô hình so với lag và lịch (xem thêm Hình 19).

In [25]:
dd = pd.DataFrame({"total": y.resample("D").sum(), "temp": ts["temp_c"].resample("D").mean()})
dd["type"] = np.where(dd.index.dayofweek >= 5, "Cuối tuần", "Ngày thường (T2–T6)")
fig = go.Figure()
for name, color in [("Ngày thường (T2–T6)", BLUE), ("Cuối tuần", ORANGE)]:
    part = dd[dd["type"] == name]
    fig.add_trace(go.Scatter(x=part["temp"], y=part["total"], mode="markers", name=name,
                             marker=dict(color=color, size=8, opacity=0.75, line=dict(color="#ffffff", width=1)),
                             hovertemplate="%{x:.1f} °C: %{y:,.0f} xe/ngày<extra></extra>"))
corr_wd = dd.loc[dd["type"] == "Ngày thường (T2–T6)", ["temp", "total"]].corr().iloc[0, 1]
fig.update_layout(title=f"Hình 13. Nhiệt độ trung bình ngày và tổng lưu lượng: tương quan yếu (r = {corr_wd:.2f} với ngày thường)",
                  xaxis_title="Nhiệt độ trung bình ngày (°C)", yaxis_title="Tổng xe/ngày", height=430)
fig.show()

### 5.6 Phân rã mùa vụ (MSTL) và tự tương quan (ACF/PACF)

Chuỗi theo giờ có **hai chu kỳ lồng nhau**: 24 giờ và 168 giờ (1 tuần). `MSTL` (Multiple Seasonal-Trend decomposition using LOESS) tách đồng thời cả hai thành phần mùa vụ, phần xu hướng và phần dư.

In [26]:
mstl = MSTL(y, periods=(24, 168), windows=(25, 169)).fit()
comp = pd.DataFrame({"Quan sát": y, "Xu hướng": mstl.trend, "Mùa vụ ngày (24h)": mstl.seasonal["seasonal_24"],
                     "Mùa vụ tuần (168h)": mstl.seasonal["seasonal_168"], "Phần dư": mstl.resid})

var_y = comp["Quan sát"].var()
share = (comp.drop(columns="Quan sát").var() / var_y * 100).rename("% phương sai của chuỗi")
strength = {k: max(0, 1 - mstl.resid.var() / (mstl.resid + comp[k]).var())
            for k in ["Mùa vụ ngày (24h)", "Mùa vụ tuần (168h)"]}
print("Độ mạnh mùa vụ (0 = không có, 1 = rất mạnh):", {k: round(v, 3) for k, v in strength.items()})
share.to_frame()

Độ mạnh mùa vụ (0 = không có, 1 = rất mạnh): {'Mùa vụ ngày (24h)': np.float64(0.953), 'Mùa vụ tuần (168h)': np.float64(0.798)}


,% phương sai của chuỗi
Xu hướng,1.15
Mùa vụ ngày (24h),79.04
Mùa vụ tuần (168h),15.36
Phần dư,3.89


**Nhận xét.** Mùa vụ ngày giải thích khoảng **79%** phương sai, mùa vụ tuần thêm khoảng **15%**, xu hướng chỉ khoảng **1%** và phần dư khoảng **4%**. Độ mạnh mùa vụ ngày là 0,95, tuần là 0,80 (thang 0–1). Chuỗi gần như không có xu hướng dài hạn, nên **không cần sai phân (differencing)** để khử xu hướng cho các mô hình hồi quy; cái cần mô hình hóa là hai chu kỳ lồng nhau.

In [27]:
sl = comp.loc["2018-03-05":"2018-03-25"]
fig = make_subplots(rows=5, cols=1, shared_xaxes=True, vertical_spacing=0.035, subplot_titles=list(comp.columns))
colors = [BLUE, ORANGE, AQUA, YELLOW, MUTED]
for i, (col, color) in enumerate(zip(comp.columns, colors), start=1):
    fig.add_trace(go.Scatter(x=sl.index, y=sl[col], mode="lines", line=dict(color=color, width=1.8), name=col,
                             hovertemplate="%{x|%a %d/%m %H:%M}: %{y:,.0f}<extra>" + col + "</extra>"), row=i, col=1)
fig.update_layout(title="Hình 14. Phân rã MSTL trên 3 tuần 05–25/03/2018: mùa vụ ngày và tuần chiếm phần lớn biến động",
                  height=820, showlegend=False)
fig.show()

In [28]:
N_ACF, N_PACF = 24 * 15, 72
acf_vals, acf_ci = acf(y, nlags=N_ACF, alpha=0.05, fft=True)
pacf_vals, pacf_ci = pacf(y, nlags=N_PACF, alpha=0.05, method="ywm")
band_acf = 1.96 / np.sqrt(len(y))

fig = make_subplots(rows=2, cols=1, vertical_spacing=0.14,
                    subplot_titles=("ACF đến 360 giờ (15 ngày)", "PACF đến 72 giờ"))
fig.add_trace(go.Bar(x=np.arange(N_ACF + 1), y=acf_vals, marker_color=BLUE, name="ACF",
                     hovertemplate="lag %{x}: %{y:.3f}<extra></extra>"), row=1, col=1)
fig.add_trace(go.Bar(x=np.arange(N_PACF + 1), y=pacf_vals, marker_color=ORANGE, name="PACF",
                     hovertemplate="lag %{x}: %{y:.3f}<extra></extra>"), row=2, col=1)
for r in (1, 2):
    for s in (band_acf, -band_acf):
        fig.add_hline(y=s, line_dash="dot", line_color=MUTED, row=r, col=1)
for lag in (24, 168, 336):
    fig.add_annotation(x=lag, y=acf_vals[lag], text=f"lag {lag}: {acf_vals[lag]:.2f}", showarrow=True,
                       arrowcolor=MUTED, ay=-30, font_color="#52514e", row=1, col=1)
fig.update_xaxes(title_text="Độ trễ (giờ)", row=2, col=1)
fig.update_layout(title="Hình 15. Tự tương quan: đỉnh ở lag 24 và lag 168 xác nhận chu kỳ ngày và tuần",
                  height=640, showlegend=False, bargap=0.3)
fig.show()

lag_tbl = pd.DataFrame({"lag (giờ)": [1, 2, 3, 24, 168], "ACF": acf_vals[[1, 2, 3, 24, 168]],
                        "PACF": [pacf_vals[k] if k <= N_PACF else np.nan for k in [1, 2, 3, 24, 168]]})
lag_tbl

,lag (giờ),ACF,PACF
0,1,0.91,0.91
1,2,0.72,-0.67
2,3,0.51,0.18
3,24,0.87,-0.10
4,168,0.94,NaN


**Gợi ý cho bước tạo đặc trưng:** ACF cao ở lag 1–3 (quán tính ngắn hạn), đỉnh lại ở **lag 24** (cùng giờ hôm qua) và còn cao hơn ở **lag 168** (cùng giờ tuần trước). PACF có ý nghĩa ở vài lag đầu và quanh lag 24 → các lag `1, 2, 3, 24, 168` cùng trung bình trượt 24 giờ là bộ đặc trưng hợp lý.

### 5.7 Phát hiện ngoại lai của `traffic_volume`: rolling mean/std so với rolling median/MAD

Áp rolling statistics trực tiếp lên chuỗi gốc không hiệu quả vì chu kỳ ngày làm độ lệch chuẩn bị "thổi phồng". Vì vậy:
1. Trừ đi **hồ sơ mùa vụ thứ × giờ** (trung vị) → phần dư $r_t$.
2. Chuẩn hóa phần dư trong cửa sổ trượt 1 tuần (168 giờ, đặt giữa), theo hai cách:
   - **Rolling mean/std:** $z_t = \dfrac{r_t - \text{mean}_{168}(r)}{\text{std}_{168}(r)}$
   - **Rolling median/MAD (robust):** $z_t = \dfrac{r_t - \text{median}_{168}(r)}{1{,}4826 \cdot \text{MAD}_{168}(r)}$ — hằng số 1,4826 đưa MAD về cùng thang với std khi dữ liệu chuẩn.
3. Đánh dấu ngoại lai khi $|z_t| > 3{,}5$ (ngưỡng Iglewicz–Hoaglin cho z-score robust).

**Câu hỏi:** cách nào phát hiện được các đoạn lưu lượng gần 0 kéo dài (đóng đường)?

In [29]:
Z_THRESHOLD = 3.5
how = ts.index.dayofweek * 24 + ts.index.hour
profile = y.groupby(how).transform("median")
resid = y - profile
win = resid.rolling(168, center=True, min_periods=84)
z_std = (resid - win.mean()) / win.std()
mad = win.apply(lambda a: np.median(np.abs(a - np.median(a))), raw=True)
z_robust = (resid - win.median()) / (1.4826 * mad)

flag_std, flag_robust = z_std.abs() > Z_THRESHOLD, z_robust.abs() > Z_THRESHOLD
ts["is_anomaly"] = flag_robust.astype(int)

# "Giờ đóng đường": ban ngày 7h–19h mà lưu lượng < 300 xe trong khi kỳ vọng > 3.000 xe.
closure = (y < 300) & (profile > 3000) & ts.index.hour.isin(range(7, 20))
compare = pd.DataFrame({
    "Phương pháp": ["Rolling mean/std", "Rolling median/MAD"],
    "Số giờ ngoại lai": [int(flag_std.sum()), int(flag_robust.sum())],
    "% số giờ": [f"{flag_std.mean():.2%}", f"{flag_robust.mean():.2%}"],
    f"Bắt được giờ đóng đường (/{int(closure.sum())})": [int((flag_std & closure).sum()), int((flag_robust & closure).sum())],
    "Trong đó là ngày lễ": [int(ts.loc[flag_std, "is_holiday"].sum()), int(ts.loc[flag_robust, "is_holiday"].sum())],
})
compare

,Phương pháp,Số giờ ngoại lai,% số giờ,Bắt được giờ đóng đường (/14),Trong đó là ngày lễ
0,Rolling mean/std,255,1.06%,0,52
1,Rolling median/MAD,1080,4.48%,14,202


In [30]:
seg = ts.loc["2016-07-04":"2016-07-31"]
fig = go.Figure()
fig.add_trace(go.Scatter(x=seg.index, y=profile.loc[seg.index], mode="lines", name="Kỳ vọng (trung vị thứ × giờ)",
                         line=dict(color=MUTED, width=1.5, dash="dot"), hovertemplate="%{y:,.0f}<extra>Kỳ vọng</extra>"))
fig.add_trace(go.Scatter(x=seg.index, y=seg["traffic_volume"], mode="lines", name="Thực tế",
                         line=dict(color=BLUE, width=1.8), hovertemplate="%{y:,.0f}<extra>Thực tế</extra>"))
for name, flag, color, symbol in [("Rolling median/MAD", flag_robust, RED, "circle"),
                                  ("Rolling mean/std", flag_std, ORANGE, "x")]:
    pts = seg[flag.loc[seg.index]]
    fig.add_trace(go.Scatter(x=pts.index, y=pts["traffic_volume"], mode="markers",
                             name=f"{name}: {len(pts)} giờ trong tháng",
                             marker=dict(color=color, size=10, symbol=symbol, line=dict(color="#ffffff", width=1.5)),
                             hovertemplate="%{x|%a %d/%m %H:%M}: %{y:,.0f}<extra>" + name + "</extra>"))
n_std, n_rob = int((flag_std & closure).sum()), int((flag_robust & closure).sum())
fig.update_layout(title=f"Hình 16. Tháng 7/2016: rolling median/MAD bắt được {n_rob}/{int(closure.sum())} giờ đóng đường ban ngày, "
                        f"rolling mean/std chỉ bắt được {n_std}",
                  yaxis_title="Xe/giờ", height=450, hovermode="x unified")
fig.show()

In [31]:
top = ts.loc[flag_robust].index.normalize().value_counts().head(10)
weather_day = ts.groupby(ts.index.normalize()).agg(holiday=("holiday", "first"),
                                                   snow_hours=("weather_main", lambda s: int((s == "Snow").sum())))
top_days = weather_day.loc[top.index].assign(**{"Số giờ ngoại lai": top.values})
top_days.index = top_days.index.strftime("%a %d/%m/%Y")
top_days.columns = ["Ngày lễ", "Số giờ có tuyết trong ngày", "Số giờ ngoại lai"]
top_days

,Ngày lễ,Số giờ có tuyết trong ngày,Số giờ ngoại lai
date_time,,,
Thu 11/01/2018,None,12,17
Mon 22/01/2018,None,19,15
Mon 19/02/2018,Washingtons Birthday,11,15
Mon 29/05/2017,Memorial Day,0,15
Sat 14/04/2018,None,20,14
Thu 23/11/2017,Thanksgiving Day,0,14
Fri 01/01/2016,New Years Day,3,14
Wed 04/07/2018,Independence Day,0,14
Sat 23/07/2016,None,0,14


**Kết luận.** Khi đường bị đóng nhiều giờ liền, chính các giờ đó làm **độ lệch chuẩn của cửa sổ phình to**, nên z-score theo mean/std nhỏ lại và ngoại lai "tự che" mình (masking effect). Median và MAD gần như không bị vài chục điểm cực đoan kéo lệch nên bắt được các đoạn này (bảng và Hình 16). Các ngày có nhiều giờ ngoại lai nhất là **ngày lễ** và **ngày bão tuyết** — đều là sự kiện thật mà mô hình cần học.

**Xử lý.**
- **Không xóa / không thay thế** giá trị mục tiêu: đây là sự kiện có thật (đóng đường, ngày lễ, bão tuyết), không phải lỗi đo như `temp = 0 K`. Sửa chúng trên tập test sẽ làm điểm đánh giá đẹp giả tạo.
- **Giữ cờ `is_anomaly`** (theo median/MAD) để phân tích sai số ở phần mô hình hóa (mô hình sai nhiều ở giờ nào?). Cờ này **không** dùng làm đặc trưng đầu vào vì được tính bằng cửa sổ đặt giữa (nhìn cả tương lai) → sẽ rò rỉ dữ liệu.

<a id="6"></a>
## 6. Chuẩn bị dữ liệu cho mô hình

### 6.1 Tạo đặc trưng

| Nhóm | Đặc trưng | Ghi chú |
|---|---|---|
| Lag | `lag_1`, `lag_2`, `lag_3`, `lag_24`, `lag_168` | Từ ACF/PACF ở mục 5.6 |
| Rolling | `roll_mean_24`, `roll_std_24` | Tính trên chuỗi đã `shift(1)` → chỉ dùng quá khứ |
| Lịch | `hour`, `dayofweek`, `month`, `is_weekend`, `is_holiday` | `hour`, `dayofweek` mã hóa thêm dạng sin/cos để 23h nằm cạnh 0h |
| Thời tiết | `temp_c`, `rain_1h`, `snow_1h`, `clouds_all`, `n_weather`, `weather_group` | `weather_group` = `weather_main`, gộp loại hiếm vào `Other` |

Thời tiết được dùng ở **cùng giờ dự báo** — giả định có dự báo thời tiết cho giờ tới (đây là giả định phổ biến trong bài toán dự báo nhu cầu, cần nêu trong báo cáo).

In [32]:
LAGS = [1, 2, 3, 24, 168]


def make_features(ts):
    f = pd.DataFrame(index=ts.index)
    yv = ts["traffic_volume"]
    for k in LAGS:
        f[f"lag_{k}"] = yv.shift(k)
    past = yv.shift(1)
    f["roll_mean_24"] = past.rolling(24).mean()
    f["roll_std_24"] = past.rolling(24).std()

    f["hour"], f["dayofweek"], f["month"] = ts.index.hour, ts.index.dayofweek, ts.index.month
    f["hour_sin"], f["hour_cos"] = np.sin(2 * np.pi * f["hour"] / 24), np.cos(2 * np.pi * f["hour"] / 24)
    f["dow_sin"], f["dow_cos"] = np.sin(2 * np.pi * f["dayofweek"] / 7), np.cos(2 * np.pi * f["dayofweek"] / 7)
    f["is_weekend"] = (f["dayofweek"] >= 5).astype(int)
    f["is_holiday"] = ts["is_holiday"]

    for c in ["temp_c", "rain_1h", "snow_1h", "clouds_all", "n_weather"]:
        f[c] = ts[c]
    counts = ts["weather_main"].value_counts()
    rare = counts[counts < 100].index
    f["weather_group"] = ts["weather_main"].where(~ts["weather_main"].isin(rare), "Other")
    f["traffic_volume"] = yv
    return f.dropna()


feat = make_features(ts)
print(f"Số dòng sau khi bỏ {len(ts) - len(feat)} giờ đầu chưa đủ lag_168: {len(feat):,}")
print("weather_group:", feat["weather_group"].value_counts().to_dict())
feat.head()

Số dòng sau khi bỏ 168 giờ đầu chưa đủ lag_168: 23,928
weather_group: {'Clear': 8218, 'Clouds': 8025, 'Rain': 2762, 'Mist': 1942, 'Snow': 1586, 'Haze': 494, 'Drizzle': 387, 'Thunderstorm': 370, 'Fog': 133, 'Other': 11}


,lag_1,lag_2,lag_3,lag_24,lag_168,roll_mean_24,roll_std_24,hour,dayofweek,month,hour_sin,hour_cos,dow_sin,dow_cos,is_weekend,is_holiday,temp_c,rain_1h,snow_1h,clouds_all,n_weather,weather_group,traffic_volume
date_time,,,,,,,,,,,,,,,,,,,,,,,
2016-01-08 00:00:00,"1,241.00","2,075.50","2,910.00",558.00,"1,513.00","3,544.38","2,016.14",0,4,1,0.00,1.00,-0.43,-0.90,0,0,0.20,0.00,0.06,75.00,4,Snow,628.00
2016-01-08 01:00:00,628.00,"1,241.00","2,075.50",323.00,"1,550.00","3,547.29","2,011.67",1,4,1,0.26,0.97,-0.43,-0.90,0,0,0.13,0.00,0.17,90.00,3,Snow,359.00
2016-01-08 02:00:00,359.00,628.00,"1,241.00",315.00,"1,134.50","3,548.79","2,009.18",2,4,1,0.50,0.87,-0.43,-0.90,0,0,0.10,0.00,0.12,90.00,1,Snow,380.00
2016-01-08 03:00:00,380.00,359.00,628.00,524.00,719.00,"3,551.50","2,004.67",3,4,1,0.71,0.71,-0.43,-0.90,0,0,0.07,0.00,0.08,90.00,2,Snow,401.00
2016-01-08 04:00:00,401.00,380.00,359.00,"1,081.00",533.00,"3,546.38","2,012.89",4,4,1,0.87,0.50,-0.43,-0.90,0,0,-0.13,0.00,0.00,90.00,2,Snow,815.00


### 6.2 Chia train / validation / test theo thời gian

**Không xáo trộn**: mô hình chỉ được học từ quá khứ để dự báo tương lai. Nếu chia ngẫu nhiên, `lag_1` của một giờ trong test chính là giá trị mục tiêu của một giờ nằm trong train → rò rỉ dữ liệu và điểm số ảo.

| Tập | Giai đoạn |
|---|---|
| Train | 08/01/2016 – 30/11/2017 |
| Validation | 01/12/2017 – 30/04/2018 |
| Test | 01/05/2018 – 30/09/2018 |

Khi tinh chỉnh siêu tham số ở phần mô hình hóa sẽ dùng `TimeSeriesSplit` trên tập train.

In [33]:
TRAIN_END, VAL_END = "2017-11-30 23:00", "2018-04-30 23:00"
train = feat.loc[:TRAIN_END]
val = feat.loc[pd.Timestamp(TRAIN_END) + pd.Timedelta(hours=1):VAL_END]
test = feat.loc[pd.Timestamp(VAL_END) + pd.Timedelta(hours=1):]
assert train.index.max() < val.index.min() and val.index.max() < test.index.min()

split_tbl = pd.DataFrame({
    "Tập": ["Train", "Validation", "Test"],
    "Từ": [d.index.min().strftime("%d/%m/%Y %H:%M") for d in (train, val, test)],
    "Đến": [d.index.max().strftime("%d/%m/%Y %H:%M") for d in (train, val, test)],
    "Số giờ": [len(d) for d in (train, val, test)],
    "Tỷ lệ": [f"{len(d) / len(feat):.1%}" for d in (train, val, test)],
    "Trung bình xe/giờ": [d["traffic_volume"].mean() for d in (train, val, test)],
    "% giờ nội suy": [f"{ts.loc[d.index, 'is_imputed'].mean():.1%}" for d in (train, val, test)],
})
split_tbl

,Tập,Từ,Đến,Số giờ,Tỷ lệ,Trung bình xe/giờ,% giờ nội suy
0,Train,08/01/2016 00:00,30/11/2017 23:00,16632,69.5%,"3,322.07",5.6%
1,Validation,01/12/2017 00:00,30/04/2018 23:00,3624,15.1%,"3,240.12",0.5%
2,Test,01/05/2018 00:00,30/09/2018 23:00,3672,15.3%,"3,358.08",0.2%


**Lưu ý.** Tập validation rơi vào mùa đông – xuân (có tuyết), còn tập test là mùa hè – thu. Lưu lượng trung bình ba tập khá gần nhau (≈ 3.240–3.360 xe/giờ), nên chênh lệch phân phối giữa các tập không lớn; tuy vậy cần nhắc trong báo cáo rằng điểm trên test phản ánh mùa hè. Tập train chứa phần lớn giờ được nội suy (5,6%), val/test gần như toàn dữ liệu thật (≤ 0,5%) → đánh giá trên dữ liệu thật.

In [34]:
fig = go.Figure()
for name, part, color in [("Train", train, BLUE), ("Validation", val, ORANGE), ("Test", test, AQUA)]:
    dm = part["traffic_volume"].resample("D").mean()
    fig.add_trace(go.Scatter(x=dm.index, y=dm.values, mode="lines", name=f"{name} ({len(part):,} giờ)",
                             line=dict(color=color, width=1.6), hovertemplate="%{x|%d/%m/%Y}: %{y:,.0f}<extra>" + name + "</extra>"))
for cut in (TRAIN_END, VAL_END):
    fig.add_vline(x=pd.Timestamp(cut), line_dash="dot", line_color="#52514e")
fig.update_layout(title="Hình 17. Chia dữ liệu theo thời gian: train → validation → test, không chồng lấn",
                  yaxis_title="Xe/giờ (trung bình ngày)", height=420, hovermode="x unified")
fig.show()

### 6.3 Encoding và scaling (fit trên **train**, áp dụng cho val/test)

- **Encoding:** `weather_group` → `OneHotEncoder(handle_unknown="ignore")` (loại thời tiết chưa thấy ở train sẽ thành vector 0, không làm lỗi). `hour`, `dayofweek` đã có bản sin/cos; `is_weekend`, `is_holiday` là nhị phân sẵn.
- **Scaling:** `StandardScaler` cho các cột số. Scaler **chỉ fit trên train** — nếu fit trên toàn bộ dữ liệu, trung bình/độ lệch chuẩn của tương lai sẽ lọt vào quá khứ. Cần cho Ridge/Linear Regression; các mô hình cây (Random Forest, Gradient Boosting) không bị ảnh hưởng bởi scaling.

In [35]:
TARGET = "traffic_volume"
CAT_COLS = ["weather_group"]
NUM_SCALE = [f"lag_{k}" for k in LAGS] + ["roll_mean_24", "roll_std_24", "temp_c", "rain_1h", "snow_1h",
                                          "clouds_all", "n_weather", "month"]
PASS_COLS = ["hour_sin", "hour_cos", "dow_sin", "dow_cos", "is_weekend", "is_holiday"]

ohe = OneHotEncoder(handle_unknown="ignore", sparse_output=False).fit(train[CAT_COLS])
scaler = StandardScaler().fit(train[NUM_SCALE])


def transform(part):
    num = pd.DataFrame(scaler.transform(part[NUM_SCALE]), columns=NUM_SCALE, index=part.index)
    cat = pd.DataFrame(ohe.transform(part[CAT_COLS]), columns=ohe.get_feature_names_out(CAT_COLS), index=part.index)
    return pd.concat([num, part[PASS_COLS], cat], axis=1)


X_train, X_val, X_test = transform(train), transform(val), transform(test)
y_train, y_val, y_test = train[TARGET], val[TARGET], test[TARGET]
print("X_train:", X_train.shape, "| X_val:", X_val.shape, "| X_test:", X_test.shape)
print("Cột one-hot:", list(ohe.get_feature_names_out(CAT_COLS)))
X_train.describe().T[["mean", "std", "min", "max"]].head(13)

X_train: (16632, 29) | X_val: (3624, 29) | X_test: (3672, 29)
Cột one-hot: ['weather_group_Clear', 'weather_group_Clouds', 'weather_group_Drizzle', 'weather_group_Fog', 'weather_group_Haze', 'weather_group_Mist', 'weather_group_Other', 'weather_group_Rain', 'weather_group_Snow', 'weather_group_Thunderstorm']


,mean,std,min,max
lag_1,0.00,1.00,-1.69,2.02
lag_2,-0.00,1.00,-1.69,2.02
lag_3,0.00,1.00,-1.69,2.02
lag_24,0.00,1.00,-1.69,2.02
lag_168,-0.00,1.00,-1.69,2.02
roll_mean_24,0.00,1.00,-6.44,1.74
roll_std_24,0.00,1.00,-5.09,2.10
temp_c,0.00,1.00,-3.36,2.11
rain_1h,-0.00,1.00,-0.08,56.76
snow_1h,0.00,1.00,-0.02,77.23


In [36]:
fig = make_subplots(rows=2, cols=2, vertical_spacing=0.18, horizontal_spacing=0.1,
                    subplot_titles=("temp_c — trước scale (°C)", "temp_c — sau scale",
                                    "lag_24 — trước scale (xe/giờ)", "lag_24 — sau scale"))
for r, col in enumerate(["temp_c", "lag_24"], start=1):
    fig.add_trace(hist_bar(train[col].to_numpy(), bins=50, color=BLUE), row=r, col=1)
    fig.add_trace(hist_bar(X_train[col].to_numpy(), bins=50, color=ORANGE), row=r, col=2)
fig.update_yaxes(title_text="Số giờ", col=1)
fig.update_layout(title="Hình 18. StandardScaler chỉ đổi thang đo (trung bình 0, độ lệch chuẩn 1), hình dạng phân phối giữ nguyên",
                  height=560, showlegend=False)
fig.show()
print(pd.DataFrame({"trước: mean": train[["temp_c", "lag_24"]].mean(), "trước: std": train[["temp_c", "lag_24"]].std(),
                    "sau: mean": X_train[["temp_c", "lag_24"]].mean(), "sau: std": X_train[["temp_c", "lag_24"]].std()}).round(3))

        trước: mean  trước: std  sau: mean  sau: std
temp_c         9.53       11.68       0.00      1.00
lag_24     3,321.64    1,962.43       0.00      1.00


In [37]:
corr_cols = [TARGET] + [f"lag_{k}" for k in LAGS] + ["roll_mean_24", "hour_sin", "hour_cos", "is_weekend",
                                                      "is_holiday", "temp_c", "rain_1h", "snow_1h", "clouds_all"]
corr = train[corr_cols].corr()
fig = go.Figure(go.Heatmap(z=corr.values, x=corr.columns, y=corr.columns, zmin=-1, zmax=1, colorscale=DIVERGING,
                           text=corr.round(2).values, texttemplate="%{text}", textfont=dict(size=10),
                           xgap=2, ygap=2, colorbar=dict(title="r"),
                           hovertemplate="%{y} × %{x}: %{z:.2f}<extra></extra>"))
fig.update_layout(title="Hình 19. Tương quan (Pearson) trên tập train: lag và giờ trong ngày mang nhiều thông tin nhất, thời tiết rất ít",
                  height=640, yaxis_autorange="reversed")
fig.show()
corr[TARGET].drop(TARGET).sort_values(key=np.abs, ascending=False).rename("r với traffic_volume").to_frame()

,r với traffic_volume
lag_168,0.95
lag_1,0.91
lag_24,0.87
hour_cos,-0.77
lag_2,0.72
lag_3,0.51
hour_sin,-0.25
is_weekend,-0.21
roll_mean_24,0.17
temp_c,0.12


**Kết luận.** `lag_168` (r = 0,95), `lag_1` (0,91) và `lag_24` (0,87) tương quan mạnh nhất với mục tiêu, tiếp theo là `hour_cos` (−0,77). Các biến thời tiết đều có |r| ≤ 0,12. Lưu ý tương quan Pearson chỉ đo quan hệ tuyến tính: ảnh hưởng của tuyết (−10,6% ở mục 5.5) là phi tuyến và hiếm, nên r nhỏ không có nghĩa là vô dụng — mô hình cây có thể khai thác được.

<a id="7"></a>
## 7. Tổng kết EDA


**Dữ liệu và làm sạch**

| Vấn đề phát hiện | Số liệu | Cách xử lý |
|---|---|---|
| Dòng trùng mốc giờ (1 giờ có nhiều hiện tượng thời tiết) | 7.629 / 48.204 dòng (15,8%) | Gộp còn 1 dòng/giờ, thêm `n_weather` |
| Giờ bị thiếu (toàn bộ 2012–2018) | 11.976 giờ (22,8%), gap dài nhất 7.386 giờ ≈ 10 tháng | Chỉ dùng 01/2016 – 09/2018 (95,8% giờ có dữ liệu, gap dài nhất 9 giờ) |
| Giờ thiếu trong giai đoạn chọn | 1.012 / 24.096 giờ (4,2%) | Reindex theo giờ, nội suy tuyến tính theo thời gian, cờ `is_imputed` |
| Lỗi cảm biến | 10 giờ `temp = 0 K`, 1 giờ `rain_1h = 9.831 mm` | Đổi thành NaN rồi nội suy |
| Nhãn ngày lễ chỉ ở 00:00 | 61 dòng | Lan nhãn cho cả 24 giờ của ngày |
| Ngoại lai của `traffic_volume` | 1.080 giờ (4,5%) theo median/MAD | Giữ nguyên giá trị (sự kiện thật), cờ `is_anomaly` để phân tích lỗi |

**Những điều EDA trả lời được (có số đối chiếu)**
1. **Chu kỳ ngày và tuần chi phối chuỗi:** MSTL cho thấy mùa vụ ngày giải thích ≈ 79% phương sai, mùa vụ tuần ≈ 15%, xu hướng chỉ ≈ 1%. Giờ đông nhất (16h) gấp ≈ 14,5 lần giờ vắng nhất (3h), trong khi chênh lệch giữa các tháng chỉ ≈ 12%.
2. **Ngày lễ giảm ≈ 28% tổng lưu lượng ngày**, tương đương cuối tuần (−26%) → cần `is_holiday`.
3. **Thời tiết chỉ quan trọng khi cực đoan:** so với cùng thứ-cùng giờ, tuyết làm giảm ≈ 10,6% và mưa/tuyết "heavy" ≈ 13,6%; các loại thời tiết khác lệch dưới 3%.
4. **Lag mạnh nhất là cùng giờ tuần trước:** `lag_168` có r = 0,95, `lag_1` 0,91, `lag_24` 0,87 → baseline naive "bằng giá trị tuần trước" sẽ là mốc so sánh khó vượt.
5. **Phát hiện ngoại lai cần thống kê robust:** rolling median/MAD bắt được 14/14 giờ đóng đường ban ngày, rolling mean/std bắt được 0/14 do hiệu ứng che lấp (masking).

**Đầu vào cho phần mô hình hóa:** `X_train` (16.632 × 29), `X_val` (3.624 × 29), `X_test` (3.672 × 29), chia theo thời gian, encoder/scaler chỉ fit trên train.

**Bước tiếp theo:** (1) baseline naive `lag_1`, `lag_24`, `lag_168` và trung bình mùa vụ; (2) Linear Regression/Ridge, Random Forest, Gradient Boosting; (3) so sánh MAE, RMSE, MAPE trên validation, chọn mô hình rồi báo cáo trên test; (4) vẽ dự báo vs thực tế trên test và phân tích sai số ở các giờ `is_anomaly`.

**Hạn chế:** chỉ có một trạm đo và một chiều đi (hướng tây); thời tiết được dùng như "biết trước" cho giờ dự báo; tập test chỉ gồm mùa hè – thu 2018.